In [ ]:
import json
import random
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import numpy as np

def _assign_session_ids(event_df: pd.DataFrame, max_gap_minutes: int = 30) -> pd.DataFrame:
    event_df = event_df.sort_values(["user_id", "event_time_dt"]).copy()
    gap = event_df.groupby("user_id")["event_time_dt"].diff()
    new_session = gap.isna() | (gap > pd.Timedelta(minutes=max_gap_minutes))
    session_idx = new_session.groupby(event_df["user_id"]).cumsum()
    event_df["session_id"] = event_df["user_id"].astype(str) + "_sess_" + session_idx.astype(int).astype(str).str.zfill(4)
    return event_df

def _random_time(start_dt: datetime, end_dt: datetime) -> datetime:
    if end_dt <= start_dt:
        return start_dt
    delta_seconds = int((end_dt - start_dt).total_seconds())
    return start_dt + timedelta(seconds=random.randint(0, delta_seconds))


def _build_user_list_pool(user_ids, seed=42):
    random.seed(seed)
    user_list_map = {}
    for uid in user_ids:
        list_count = random.randint(2, 6)
        user_list_map[uid] = [f"list_{uid}_{i:02d}" for i in range(1, list_count + 1)]
    return user_list_map


def generate_dim_task(
    user_df: pd.DataFrame,
    num_tasks=180000,
    seed=42,
    start_date="2026-07-01",
    end_date="2026-08-29"
):
    rng = np.random.default_rng(seed)

    assert 120000 <= num_tasks <= 250000, "dim_task 行数建议在 120,000 ~ 250,000"

    start_dt = np.datetime64(f"{start_date}T00:00:00")
    end_dt = np.datetime64(f"{end_date}T23:59:59")
    total_seconds = int((np.datetime64(end_dt) - np.datetime64(start_dt)).astype("timedelta64[s]").astype(int))

    user_ids = user_df["user_id"].to_numpy()
    task_id = np.char.mod("task_%08d", np.arange(1, num_tasks + 1))
    user_id = rng.choice(user_ids, size=num_tasks)

    user_list_map = _build_user_list_pool(user_ids, seed=seed)
    list_id = np.array([rng.choice(user_list_map[u]) for u in user_id], dtype=object)

    create_offsets = rng.integers(0, total_seconds + 1, size=num_tasks)
    create_ts = start_dt + create_offsets.astype("timedelta64[s]")
    create_time = pd.to_datetime(create_ts)

    has_due_date = rng.random(num_tasks) < 0.68
    due_date = np.full(num_tasks, None, dtype=object)
    due_start = create_time.floor("D").to_numpy(dtype="datetime64[D]")
    due_end = np.minimum(due_start + np.timedelta64(30, "D"), np.datetime64(end_dt, "D"))
    span_days = (due_end - due_start).astype(int)
    due_offsets = rng.integers(0, span_days + 1)
    due_ts = due_start + due_offsets.astype("timedelta64[D]")

    if has_due_date.any():
        due_date[has_due_date] = pd.to_datetime(due_ts[has_due_date]).strftime("%Y-%m-%d")

    priorities = ["high", "medium", "low"]
    priority_weights = [0.18, 0.32, 0.50]
    priority = rng.choice(priorities, size=num_tasks, p=priority_weights)

    has_subtask = rng.random(num_tasks) < 0.36
    has_reminder = rng.random(num_tasks) < 0.42

    priority_score = np.select(
        [priority == "high", priority == "medium", priority == "low"],
        [0.18, 0.05, -0.10],
        default=0.0
    )

    due_day = np.full(num_tasks, 30, dtype=int)
    if has_due_date.any():
        due_day[has_due_date] = (due_ts[has_due_date] - due_start[has_due_date]).astype(int)

    due_date_score = np.where(
        has_due_date,
        np.where(due_day <= 7, 0.18, 0.08),
        -0.05
    )

    reminder_score = np.where(has_reminder, 0.08, -0.03)

    completion_prob = np.clip(
        0.25 + priority_score + due_date_score + reminder_score,
        0.05,
        0.92
    )

    is_completed = rng.random(num_tasks) < completion_prob

    complete_time = np.full(num_tasks, None, dtype=object)
    completed_idx = np.flatnonzero(is_completed)
    if len(completed_idx) > 0:
        end_ts = np.datetime64(end_dt)
        delta_seconds = ((end_ts - create_ts[completed_idx])
                         .astype("timedelta64[s]").astype(int))
        complete_offsets = rng.integers(0, delta_seconds + 1)
        complete_ts = create_ts[completed_idx] + complete_offsets.astype("timedelta64[s]")
        complete_time[completed_idx] = pd.to_datetime(complete_ts).strftime("%Y-%m-%d %H:%M:%S")

    verbs = np.array(["Plan", "Review", "Write", "Check", "Refactor", "Prepare", "Sync", "Update", "Clean", "Analyze"])
    objects = np.array(["report", "meeting notes", "todo list", "daily plan", "weekly summary",
                        "data check", "dashboard", "sql script", "user feedback", "bug list"])
    task_title = np.char.add(
        rng.choice(verbs, size=num_tasks),
        " " + rng.choice(objects, size=num_tasks)
    )

    df = pd.DataFrame({
        "task_id": task_id,
        "user_id": user_id,
        "list_id": list_id,
        "task_title": task_title,
        "priority": priority,
        "due_date": due_date,
        "create_time": create_time.strftime("%Y-%m-%d %H:%M:%S"),
        "complete_time": complete_time,
        "is_completed": is_completed,
        "has_subtask": has_subtask,
        "has_reminder": has_reminder,
        "task_status": np.where(
            is_completed,
            np.where(rng.random(num_tasks) < 0.90, "completed", "deleted"),
            np.where(rng.random(num_tasks) < 0.88, "active", "deleted")
        )
    })

    return df[
        [
            "task_id", "user_id", "list_id", "task_title", "priority", "due_date",
            "create_time", "complete_time", "is_completed", "has_subtask",
            "has_reminder", "task_status"
        ]
    ]


def generate_ods_todo_event_log(
    user_df: pd.DataFrame,
    task_df: pd.DataFrame,
    num_events=600000,
    seed=42,
    start_date="2026-07-01",
    end_date="2026-08-29"
):
    rng = np.random.default_rng(seed)

    assert 400000 <= num_events <= 800000, "ods_todo_event_log 行数建议在 400,000 ~ 800,000"
    assert num_events <= 1000000, "ods_todo_event_log 建议不超过 1,000,000"

    start_dt = np.datetime64(f"{start_date}T00:00:00")
    end_dt = np.datetime64(f"{end_date}T23:59:59")
    total_seconds = int((end_dt - start_dt).astype("timedelta64[s]").astype(int))

    task_df2 = task_df.copy()
    task_df2["create_time_dt"] = pd.to_datetime(task_df2["create_time"])
    task_df2["complete_time_dt"] = pd.to_datetime(task_df2["complete_time"], errors="coerce")

    device_os_map = {
        "mobile": {
            "android": 0.58,
            "ios": 0.42
        },
        "tablet": {
            "android": 0.35,
            "ios": 0.65
        },
        "desktop": {
            "windows": 0.72,
            "macos": 0.28
        }
    }
    device_types = list(device_os_map.keys())

    def sample_device_os(size):
        device_type_arr = rng.choice(
            device_types,
            size=size,
            p=[0.50, 0.15, 0.35]
        )

        os_arr = np.empty(size, dtype=object)

        for device_type in device_types:
            mask = device_type_arr == device_type

            os_names = list(device_os_map[device_type].keys())
            os_probs = list(device_os_map[device_type].values())

            os_arr[mask] = rng.choice(
                os_names,
                size=mask.sum(),
                p=os_probs
            )

        return device_type_arr, os_arr
    
    app_versions = ["3.0.1", "3.1.0", "3.1.2", "3.2.0", "3.2.1"]
    network_types = ["wifi", "4g", "5g"]

    # create_task
    create_n = len(task_df2)
    device_type_arr, os_arr = sample_device_os(create_n)
    create_attrs = [
        json.dumps({
            "priority": p,
            "due_date": d,
            "has_reminder": bool(r),
            "has_subtask": bool(s),
            "action_duration": int(a)
        }, ensure_ascii=False)
        for p, d, r, s, a in zip(
            task_df2["priority"],
            task_df2["due_date"],
            task_df2["has_reminder"],
            task_df2["has_subtask"],
            rng.integers(3, 121, size=create_n)
        )
    ]
    create_events = pd.DataFrame({
        "log_id": np.char.mod("log_%010d", np.arange(1, create_n + 1)),
        "user_id": task_df2["user_id"].to_numpy(),
        "session_id": np.full(create_n, None, dtype=object),
        "event_time": task_df2["create_time"].to_numpy(),
        "event_type": np.full(create_n, "create_task", dtype=object),
        "task_id": task_df2["task_id"].to_numpy(),
        "list_id": task_df2["list_id"].to_numpy(),
        "attributes": create_attrs,
        "device_type": device_type_arr,
        "os": os_arr,
        "app_version": rng.choice(app_versions, size=create_n),
        "network_type": rng.choice(network_types, size=create_n),
    })

    # complete_task
    completed = task_df2["is_completed"]
    complete_n = completed.sum()
    complete_device_type, complete_os = sample_device_os(complete_n)
    complete_events = pd.DataFrame({
        "log_id": np.char.mod("log_%010d", np.arange(create_n + 1, create_n + complete_n + 1)),
        "user_id": task_df2.loc[completed, "user_id"].to_numpy(),
        "session_id": np.full(complete_n, None, dtype=object),
        "event_time": task_df2.loc[completed, "complete_time"].to_numpy(),
        "event_type": np.full(complete_n, "complete_task", dtype=object),
        "task_id": task_df2.loc[completed, "task_id"].to_numpy(),
        "list_id": task_df2.loc[completed, "list_id"].to_numpy(),
        "attributes": [
            json.dumps({
                "priority": p,
                "due_date": d,
                "action_duration": int(a)
            }, ensure_ascii=False)
            for p, d, a in zip(
                task_df2.loc[completed, "priority"],
                task_df2.loc[completed, "due_date"],
                rng.integers(1, 181, size=complete_n)
            )
        ],
        "device_type": complete_device_type,
        "os": complete_os,
        "app_version": rng.choice(app_versions, size=complete_n),
        "network_type": rng.choice(network_types, size=complete_n),
    })

    # delete_task
    deleted = task_df2["task_status"] == "deleted"
    delete_n = deleted.sum()
    delete_device_type, delete_os = sample_device_os(delete_n)
    delete_create_ts = task_df2.loc[deleted, "create_time_dt"].to_numpy(dtype="datetime64[s]")
    delete_delta = ((end_dt - delete_create_ts).astype("timedelta64[s]").astype(int))
    delete_offsets = rng.integers(0, delete_delta + 1)
    delete_time = delete_create_ts + delete_offsets.astype("timedelta64[s]")
    delete_events = pd.DataFrame({
        "log_id": np.char.mod("log_%010d", np.arange(create_n + complete_n + 1,
                                                     create_n + complete_n + delete_n + 1)),
        "user_id": task_df2.loc[deleted, "user_id"].to_numpy(),
        "session_id": np.full(delete_n, None, dtype=object),
        "event_time": pd.to_datetime(delete_time).strftime("%Y-%m-%d %H:%M:%S"),
        "event_type": np.full(delete_n, "delete_task", dtype=object),
        "task_id": task_df2.loc[deleted, "task_id"].to_numpy(),
        "list_id": task_df2.loc[deleted, "list_id"].to_numpy(),
        "attributes": [
            json.dumps({
                "priority": p,
                "due_date": d,
                "action_duration": int(a)
            }, ensure_ascii=False)
            for p, d, a in zip(
                task_df2.loc[deleted, "priority"],
                task_df2.loc[deleted, "due_date"],
                rng.integers(1, 181, size=delete_n)
            )
        ],
        "device_type": delete_device_type,
        "os": delete_os,
        "app_version": rng.choice(app_versions, size=delete_n),
        "network_type": rng.choice(network_types, size=delete_n),
    })

    # remaining events
    base_events = num_events - (create_n + complete_n + delete_n)
    if base_events < 0:
        raise ValueError("num_events 小于必须生成的 create/complete/delete 事件数量")

    user_ids = user_df["user_id"].to_numpy()
    sampled_idx = rng.integers(0, len(task_df2), size=base_events)
    sampled = task_df2.iloc[sampled_idx].reset_index(drop=True)

    other_event_types = ["app_launch", "app_close","edit_task", "set_due_date", "set_priority",
                         "search_task", "view_list", "use_feature"]
    weights = [0.18, 0.06, 0.15, 0.12, 0.12, 0.16, 0.16, 0.05]
    event_type_arr = rng.choice(other_event_types, size=base_events, p=weights)

    event_time = np.empty(base_events, dtype=object)
    attrs = [None] * base_events
    user_id_arr = rng.choice(user_ids, size=base_events)

    device_arr, os_arr = sample_device_os(base_events)
    version_arr = rng.choice(app_versions, size=base_events)
    network_arr = rng.choice(network_types, size=base_events)
    action_duration_arr = rng.integers(1, 181, size=base_events)

    edit_mask = np.isin(event_type_arr, ["edit_task", "set_due_date", "set_priority"])
    event_time[~edit_mask] = (
        pd.to_datetime(start_dt) + pd.to_timedelta(rng.integers(0, total_seconds + 1, size=(~edit_mask).sum()), unit="s")
    ).strftime("%Y-%m-%d %H:%M:%S")

    if edit_mask.any():
        edit_create_ts = sampled.loc[edit_mask, "create_time_dt"].to_numpy(dtype="datetime64[s]")
        edit_end_ts = np.where(
            sampled.loc[edit_mask, "complete_time_dt"].notna(),
            sampled.loc[edit_mask, "complete_time_dt"].to_numpy(dtype="datetime64[s]"),
            end_dt
        )
        edit_delta = ((edit_end_ts - edit_create_ts).astype("timedelta64[s]").astype(int))
        edit_offsets = rng.integers(0, edit_delta + 1)
        event_time[edit_mask] = (
            pd.to_datetime(edit_create_ts + edit_offsets.astype("timedelta64[s]"))
        ).strftime("%Y-%m-%d %H:%M:%S")

    for i, et in enumerate(event_type_arr):
        if et in {"edit_task", "set_due_date", "set_priority"}:
            row = sampled.iloc[i]
            base = {
                "action_duration": int(action_duration_arr[i]),
                "priority": row["priority"],
                "due_date": row["due_date"],
                "has_reminder": bool(row["has_reminder"]),
                "has_subtask": bool(row["has_subtask"]),
            }
            if et == "set_priority":
                base["new_priority"] = rng.choice(["high", "medium", "low"])
            elif et == "set_due_date":
                base["new_due_date"] = (
                    pd.to_datetime(event_time[i]).date() + pd.Timedelta(days=rng.integers(0, 15))
                ).isoformat()
            attrs[i] = json.dumps(base, ensure_ascii=False)
        elif et == "search_task":
            attrs[i] = json.dumps({
                "action_duration": int(action_duration_arr[i]),
                "query": rng.choice(["work", "today", "urgent", "meeting", "report", "home"]),
                "result_count": int(rng.integers(0, 51))
            }, ensure_ascii=False)
        elif et == "use_feature":
            attrs[i] = json.dumps({
                "action_duration": int(action_duration_arr[i]),
                "feature_name": rng.choice(["kanban_view", "smart_sort", "batch_edit", "focus_mode"])
            }, ensure_ascii=False)
        elif et == "app_close":
            attrs[i] = json.dumps({
                "action_duration": int(action_duration_arr[i]),
                "close_reason": "user_exit"
            }, ensure_ascii=False)
        else:
            attrs[i] = json.dumps({"action_duration": int(action_duration_arr[i])}, ensure_ascii=False)

    remaining_events = pd.DataFrame({
        "log_id": np.char.mod("log_%010d", np.arange(create_n + complete_n + delete_n + 1, num_events + 1)),
        "user_id": user_id_arr,
        "session_id": np.full(base_events, None, dtype=object),
        "event_time": event_time,
        "event_type": event_type_arr,
        "task_id": np.where(edit_mask, sampled["task_id"].to_numpy(), None),
        "list_id": np.where(edit_mask, sampled["list_id"].to_numpy(), None),
        "attributes": attrs,
        "device_type": device_arr,
        "os": os_arr,
        "app_version": version_arr,
        "network_type": network_arr,
    })

    event_df = pd.concat([create_events, complete_events, delete_events, remaining_events], ignore_index=True)
    event_df["event_time_dt"] = pd.to_datetime(event_df["event_time"])
    event_df = _assign_session_ids(event_df, max_gap_minutes=30)
    event_df["event_date"] = event_df["event_time_dt"].dt.strftime("%Y-%m-%d")

    return event_df.sort_values("event_time_dt").reset_index(drop=True)[
        [
            "log_id", "user_id", "session_id", "event_time",
            "event_type", "task_id", "list_id", "attributes",
            "device_type", "os", "app_version", "network_type",
            "event_date"
        ]
    ]



# 主程序
user_path="output_data/01_dim_user.parquet"
task_rows=180000
event_rows=600000
seed=42
start_date="2026-07-01"
end_date="2026-08-29"

user_df = pd.read_parquet(user_path)

task_df = generate_dim_task(
    user_df=user_df,
    num_tasks=task_rows,
    seed=seed,
    start_date=start_date,
    end_date=end_date
)

event_df = generate_ods_todo_event_log(
    user_df=user_df,
    task_df=task_df,
    num_events=event_rows,
    seed=seed,
    start_date=start_date,
    end_date=end_date
)

output_dir = Path("output_data")
output_dir.mkdir(exist_ok=True)

# 1) 任务维度单文件 parquet
task_path = output_dir / "03_dim_task.parquet"
task_df.to_parquet(task_path, index=False)

# 2) 行为日志单文件 parquet
event_flat_path = output_dir / "03_ods_todo_event_log.parquet"
event_df.drop(columns=["event_date"]).to_parquet(event_flat_path, index=False)

# 3) 行为日志按天分区 parquet（每天一个分区）
event_partition_path = output_dir / "ods_todo_event_log"
event_df.to_parquet(event_partition_path, index=False, partition_cols=["event_date"])

# 校验指标
n_users = user_df["user_id"].nunique()
n_days = (datetime.fromisoformat(end_date) - datetime.fromisoformat(start_date)).days + 1

avg_tasks_per_user = len(task_df) / n_users
completion_rate = task_df["is_completed"].mean()
avg_events_per_user_per_day = len(event_df) / (n_users * n_days)

print(f"✅ dim_task saved: {task_path} ({len(task_df)} rows)")
print(f"✅ ods_todo_event_log flat parquet saved: {event_flat_path} ({len(event_df)} rows)")
print(f"✅ ods_todo_event_log partition parquet saved: {event_partition_path}")
print(f"✅ avg tasks per user: {avg_tasks_per_user:.2f}")
print(f"✅ task completion rate: {completion_rate:.2%}")
print(f"✅ avg events per user per day: {avg_events_per_user_per_day:.3f}")

print("\nEvent type distribution:")
print(event_df["event_type"].value_counts(normalize=True).round(4).to_string())

✅ dim_task saved: output_data\03_dim_task.parquet (180000 rows)
✅ ods_todo_event_log flat parquet saved: output_data\03_ods_todo_event_log.parquet (600000 rows)
✅ ods_todo_event_log partition parquet saved: output_data\ods_todo_event_log
✅ avg tasks per user: 18.00
✅ task completion rate: 33.15%
✅ avg events per user per day: 1.000

Event type distribution:
event_type
create_task      0.3000
app_launch       0.1013
complete_task    0.0995
search_task      0.0907
view_list        0.0907
edit_task        0.0851
set_due_date     0.0682
set_priority     0.0681
delete_task      0.0342
app_close        0.0340
use_feature      0.0283
